In [1]:
import StableDiffusion.ModelConverter
from StableDiffusion.DiffusionProcess import DiffusionProcess
device = 'cuda'
idleDevice = 'cpu'
diffusionDict = StableDiffusion.ModelConverter.load_from_standard_weights(input_file='../models/sd15models/v1-5-pruned-emaonly.ckpt',\
                                                            device = 'cuda')
clipWeights=diffusionDict['clip']
diffusionWeights = diffusionDict['diffusion']
vaeEncoderWeights = diffusionDict['encoder']
vaeDecoderWeights = diffusionDict['decoder']



import torch 
import StableDiffusion.VaeEncoder 
import StableDiffusion.VaeDecoder
import StableDiffusion.ClipEncoder
import StableDiffusion.DiffusionProcess
import importlib
importlib.reload(StableDiffusion.VaeEncoder)
importlib.reload(StableDiffusion.VaeDecoder)
importlib.reload(StableDiffusion.ClipEncoder)
importlib.reload(StableDiffusion.DiffusionProcess)
from StableDiffusion.VaeDecoder import VaeDecoder
from StableDiffusion.VaeEncoder import VaeEncoder
from StableDiffusion.ClipEncoder import ClipEncoder
from StableDiffusion.DiffusionProcess import DiffusionProcess
#clipEncoder = ClipEncoder().to(device)
#vaeEncoder = VaeEncoder().to(device)
#vaeDecoder = VaeDecoder().to(device)
diffusionProcess = DiffusionProcess().to(device)
#clipEncoder.load_state_dict(clipWeights,strict=True)
#vaeEncoder.load_state_dict(vaeEncoderWeights ,strict=True)
#vaeDecoder.load_state_dict(vaeDecoderWeights,strict=True)
diffusionProcess.load_state_dict(diffusionWeights,strict=True)
#clipName = clipEncoder.__class__.__name__
#print(clipName)



/home/aistudio/external-libraries/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<All keys matched successfully>

In [3]:
import StableDiffusion.ControlnetSDUnet
import StableDiffusion.ControlnetSD
import StableDiffusion.DiffusionProcessControlnet
import importlib 
importlib.reload(StableDiffusion.ControlnetSDUnet)
importlib.reload(StableDiffusion.ControlnetSD)
importlib.reload(StableDiffusion.DiffusionProcessControlnet)
from StableDiffusion.ControlnetSDUnet import ControlnetSDUnet
from StableDiffusion.ControlnetSD import ControlnetSD
from StableDiffusion.DiffusionProcessControlnet import DiffusionProcessControlnet

from StableDiffusion.Utils import Utils
dpControlnet = DiffusionProcessControlnet().to(device)
missingKeys, unexpectedKeys = dpControlnet.load_state_dict(diffusionWeights,strict = False)

In [3]:
from StableDiffusion.UnetDenoise import UnetDenoise
from StableDiffusion.TimeEmbedding import TimeEmbedding
unetDenoise = UnetDenoise().to(device)
timeEmbLayer = TimeEmbedding().to(device)

In [4]:
import StableDiffusion.ControlnetSDUnet
import StableDiffusion.ControlnetSD
import StableDiffusion.DiffusionProcessControlnet
import importlib 
importlib.reload(StableDiffusion.ControlnetSDUnet)
importlib.reload(StableDiffusion.ControlnetSD)
importlib.reload(StableDiffusion.DiffusionProcessControlnet)
from StableDiffusion.ControlnetSDUnet import ControlnetSDUnet
from StableDiffusion.ControlnetSD import ControlnetSD
from StableDiffusion.DiffusionProcessControlnet import DiffusionProcessControlnet

from StableDiffusion.Utils import Utils


device = 'cuda'

controlnetSD = ControlnetSD(unetDenoise)
controlnetSDUnet = ControlnetSDUnet(unetDenoise,timeEmbLayer)

In [25]:
for name,param in controlnetSD.named_parameters():
    print(name,param.shape,param.requires_grad)

unet.encoders.0.0.weight torch.Size([320, 4, 3, 3]) False
unet.encoders.0.0.bias torch.Size([320]) False
unet.encoders.1.0.groupnorm_feature.weight torch.Size([320]) False
unet.encoders.1.0.groupnorm_feature.bias torch.Size([320]) False
unet.encoders.1.0.conv_feature.weight torch.Size([320, 320, 3, 3]) False
unet.encoders.1.0.conv_feature.bias torch.Size([320]) False
unet.encoders.1.0.linear_time.weight torch.Size([320, 1280]) False
unet.encoders.1.0.linear_time.bias torch.Size([320]) False
unet.encoders.1.0.groupnorm_merged.weight torch.Size([320]) False
unet.encoders.1.0.groupnorm_merged.bias torch.Size([320]) False
unet.encoders.1.0.conv_merged.weight torch.Size([320, 320, 3, 3]) False
unet.encoders.1.0.conv_merged.bias torch.Size([320]) False
unet.encoders.1.1.groupnorm.weight torch.Size([320]) False
unet.encoders.1.1.groupnorm.bias torch.Size([320]) False
unet.encoders.1.1.conv_input.weight torch.Size([320, 320, 1, 1]) False
unet.encoders.1.1.conv_input.bias torch.Size([320]) Fals

In [14]:
import torch

def convert_ldm_to_diffusers(state_dict):
    new_dict = {}
    
    # Mapping for standard blocks
    key_map = {
        "in_layers.0": "groupnorm_feature",
        "in_layers.2": "conv_feature",
        "emb_layers.1": "linear_time",
        "out_layers.0": "groupnorm_merged",
        "out_layers.3": "conv_merged",
        "transformer_blocks.0.norm1": "layernorm_1",
        "transformer_blocks.0.norm2": "layernorm_2",
        "transformer_blocks.0.norm3": "layernorm_3",
        "ff.net.0.proj": "linear_geglu_1",
        "ff.net.2": "linear_geglu_2",
        "proj_in": "conv_input",
        "proj_out": "conv_output",
        "skip_connection": "residual_layer"
    }

    for key, weight in state_dict.items():
        # Remove prefix
        k = key.replace("control_model.", "")
        
        # 1. Handle Self-Attention (attn1) concatenation
        if "attn1.to_" in k:
            continue
            
        # 2. Rename standard layers
        for old, new in key_map.items():
            k = k.replace(old, new)
            
        # 3. Clean up specific naming patterns
        k = k.replace("attn1.to_out.0", "attention_1.out_proj")
        k = k.replace("attn2.to_out.0", "attention_2.out_proj")
        
        # Cross-Attention naming
        k = k.replace("attn2.to_q", "attention_2.q_proj")
        k = k.replace("attn2.to_k", "attention_2.k_proj")
        k = k.replace("attn2.to_v", "attention_2.v_proj")
        
        new_dict[k] = weight

    # 4. Perform Concatenation for Self-Attention (attn1)
    # Target all blocks that contain transformer_blocks.0.attn1
    for i in range(12): # input_blocks
        for j in [1]:  # transformer_blocks index
            prefix = f"input_blocks.{i}.{j}.transformer_blocks.0.attn1"
            q = state_dict.get(f"control_model.input_blocks.{i}.{j}.transformer_blocks.0.attn1.to_q.weight")
            k = state_dict.get(f"control_model.input_blocks.{i}.{j}.transformer_blocks.0.attn1.to_k.weight")
            v = state_dict.get(f"control_model.input_blocks.{i}.{j}.transformer_blocks.0.attn1.to_v.weight")
            
            if q is not None:
                new_dict[f"input_blocks.{i}.{j}.attention_1.in_proj.weight"] = torch.cat([q, k, v], dim=0)

    return new_dict

# Usage
controlDict = convert_ldm_to_diffusers(state_dict)

for k, v in controlDict.items():
    print(k, v.shape)

time_embed.0.weight torch.Size([1280, 320])
time_embed.0.bias torch.Size([1280])
time_embed.2.weight torch.Size([1280, 1280])
time_embed.2.bias torch.Size([1280])
input_blocks.0.0.weight torch.Size([320, 4, 3, 3])
input_blocks.0.0.bias torch.Size([320])
input_blocks.1.0.groupnorm_feature.weight torch.Size([320])
input_blocks.1.0.groupnorm_feature.bias torch.Size([320])
input_blocks.1.0.conv_feature.weight torch.Size([320, 320, 3, 3])
input_blocks.1.0.conv_feature.bias torch.Size([320])
input_blocks.1.0.linear_time.weight torch.Size([320, 1280])
input_blocks.1.0.linear_time.bias torch.Size([320])
input_blocks.1.0.groupnorm_merged.weight torch.Size([320])
input_blocks.1.0.groupnorm_merged.bias torch.Size([320])
input_blocks.1.0.conv_merged.weight torch.Size([320, 320, 3, 3])
input_blocks.1.0.conv_merged.bias torch.Size([320])
input_blocks.1.1.norm.weight torch.Size([320])
input_blocks.1.1.norm.bias torch.Size([320])
input_blocks.1.1.conv_input.weight torch.Size([320, 320, 1, 1])
input_bl

In [23]:
import torch
import StableDiffusion.ControlnetModelConverter
import importlib
importlib.reload(StableDiffusion.ControlnetModelConverter)
from StableDiffusion.ControlnetModelConverter import ControlnetModelConverter
filePath ="../models/ControlNet-v1-1/control_v11p_sd15_canny.pth" 
state_dict = torch.load(filePath, map_location="cpu")

newDict = ControlnetModelConverter(filePath)
for key,value in state_dict.items():
    print(key,value.shape)
#print(type(state_dict))
#print(state_dict.keys())

control_model.time_embed.0.weight torch.Size([1280, 320])
control_model.time_embed.0.bias torch.Size([1280])
control_model.time_embed.2.weight torch.Size([1280, 1280])
control_model.time_embed.2.bias torch.Size([1280])
control_model.input_blocks.0.0.weight torch.Size([320, 4, 3, 3])
control_model.input_blocks.0.0.bias torch.Size([320])
control_model.input_blocks.1.0.in_layers.0.weight torch.Size([320])
control_model.input_blocks.1.0.in_layers.0.bias torch.Size([320])
control_model.input_blocks.1.0.in_layers.2.weight torch.Size([320, 320, 3, 3])
control_model.input_blocks.1.0.in_layers.2.bias torch.Size([320])
control_model.input_blocks.1.0.emb_layers.1.weight torch.Size([320, 1280])
control_model.input_blocks.1.0.emb_layers.1.bias torch.Size([320])
control_model.input_blocks.1.0.out_layers.0.weight torch.Size([320])
control_model.input_blocks.1.0.out_layers.0.bias torch.Size([320])
control_model.input_blocks.1.0.out_layers.3.weight torch.Size([320, 320, 3, 3])
control_model.input_block

In [24]:

for key,value in newDict.items():
    print(key,value.shape)

time_embed.0.weight torch.Size([1280, 320])
time_embed.0.bias torch.Size([1280])
time_embed.2.weight torch.Size([1280, 1280])
time_embed.2.bias torch.Size([1280])
input_blocks.0.0.weight torch.Size([320, 4, 3, 3])
input_blocks.0.0.bias torch.Size([320])
input_blocks.1.0.groupnorm_feature.weight torch.Size([320])
input_blocks.1.0.groupnorm_feature.bias torch.Size([320])
input_blocks.1.0.conv_feature.weight torch.Size([320, 320, 3, 3])
input_blocks.1.0.conv_feature.bias torch.Size([320])
input_blocks.1.0.linear_time.weight torch.Size([320, 1280])
input_blocks.1.0.linear_time.bias torch.Size([320])
input_blocks.1.0.groupnorm_merged.weight torch.Size([320])
input_blocks.1.0.groupnorm_merged.bias torch.Size([320])
input_blocks.1.0.conv_merged.weight torch.Size([320, 320, 3, 3])
input_blocks.1.0.conv_merged.bias torch.Size([320])
input_blocks.1.1.norm.weight torch.Size([320])
input_blocks.1.1.norm.bias torch.Size([320])
input_blocks.1.1.conv_input.weight torch.Size([320, 320, 1, 1])
input_bl

In [3]:
latent = torch.randn(4,4,64,64,device=device)
context = torch.randn(4,77,768,device=device)
controlHint = torch.randn(4,3,512,512,device=device)
timeSteps = torch.randint(0,1000,(4,),device=device)
#print(timeSteps)
timeEmb320 = Utils.getTimeEmbeddingBatchTorch(timeSteps)
outPredictedNoise = dpControlnet(latent,context,timeSteps,controlHint)

lantex.shape after unet torch.Size([4, 4, 64, 64])
latent shape torch.Size([4, 320, 64, 64]) cannyLatent shape torch.Size([4, 320, 64, 64])


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 16.00 GiB of which 1.29 GiB is free. Process 2879059 has 14.71 GiB memory in use. Of the allocated memory 12.04 GiB is allocated by PyTorch, and 1.53 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
import StableDiffusion.ControlnetSDUnet
import StableDiffusion.ControlnetSD
import StableDiffusion.DiffusionProcessControlnet
import importlib 
importlib.reload(StableDiffusion.ControlnetSDUnet)
importlib.reload(StableDiffusion.ControlnetSD)
importlib.reload(StableDiffusion.DiffusionProcessControlnet)
from StableDiffusion.ControlnetSDUnet import ControlnetSDUnet
from StableDiffusion.ControlnetSD import ControlnetSD
from StableDiffusion.DiffusionProcessControlnet import DiffusionProcessControlnet

from StableDiffusion.Utils import Utils

unetOriginal = diffusionProcess.unet
device = 'cuda'
unetDenoise = unetOriginal.to(device)
controlnetSD = ControlnetSD(unetDenoise)
controlnetSDUnet = ControlnetSDUnet(unetDenoise)
dpc = DiffusionProcessControlnet(unetDenoise)
unetDenoise.to(device)
controlnetSD.to(device)
controlnetSDUnet.to(device)
dpc.to(device)
Utils.showModelDevice(controlnetSD)
Utils.showModelDevice(unetDenoise)
Utils.showModelDevice(controlnetSDUnet)

for name,param in controlnetSD.named_parameters():
    print(name,param.shape,param.requires_grad)


for name,param in controlnetSDUnet.named_parameters():
    print(name,param.shape,param.requires_grad)

✅ Model is entirely on: cuda:0
✅ Model is entirely on: cuda:0
✅ Model is entirely on: cuda:0
unet.encoders.0.0.weight torch.Size([320, 4, 3, 3]) False
unet.encoders.0.0.bias torch.Size([320]) False
unet.encoders.1.0.groupnorm_feature.weight torch.Size([320]) False
unet.encoders.1.0.groupnorm_feature.bias torch.Size([320]) False
unet.encoders.1.0.conv_feature.weight torch.Size([320, 320, 3, 3]) False
unet.encoders.1.0.conv_feature.bias torch.Size([320]) False
unet.encoders.1.0.linear_time.weight torch.Size([320, 1280]) False
unet.encoders.1.0.linear_time.bias torch.Size([320]) False
unet.encoders.1.0.groupnorm_merged.weight torch.Size([320]) False
unet.encoders.1.0.groupnorm_merged.bias torch.Size([320]) False
unet.encoders.1.0.conv_merged.weight torch.Size([320, 320, 3, 3]) False
unet.encoders.1.0.conv_merged.bias torch.Size([320]) False
unet.encoders.1.1.groupnorm.weight torch.Size([320]) False
unet.encoders.1.1.groupnorm.bias torch.Size([320]) False
unet.encoders.1.1.conv_input.weigh

In [3]:
latent = torch.randn(4,4,64,64,device=device)
context = torch.randn(4,77,768,device=device)
controlHint = torch.randn(4,3,512,512,device=device)
timeSteps = torch.randint(0,1000,(4,),device=device)
#print(timeSteps)
timeEmb320 = Utils.getTimeEmbeddingBatchTorch(timeSteps)

#print(f'timeEmb320.shape {timeEmb320.shape} timeEmb320.device {timeEmb320.device}')
controlnetOutputs = controlnetSD(latent,context,timeSteps,controlHint)
print(f'controlnetOutputs length {len(controlnetOutputs)}')
#print(f'controlnetOutputs {controlnetOutputs} length {len(controlnetOutputs)}')
controlSDUnetOutput = controlnetSDUnet(latent,context,timeSteps,controlnetOutputs)
print(f'controlSDUnetOutput.shape {controlSDUnetOutput.shape}')

latent shape torch.Size([4, 320, 64, 64]) cannyLatent shape torch.Size([4, 320, 64, 64])
controlnetOutputs length 13
controlSDUnetOutput.shape torch.Size([4, 320, 64, 64])


In [4]:
dpcOutputs = dpc(latent,context,timeSteps,controlHint)
print(f'dpcOutputs.shape {dpcOutputs.shape}')

lantex.shape after unet torch.Size([4, 4, 64, 64])
latent shape torch.Size([4, 320, 64, 64]) cannyLatent shape torch.Size([4, 320, 64, 64])


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 31.73 GiB of which 1.56 GiB is free. Process 1649004 has 30.17 GiB memory in use. Of the allocated memory 27.84 GiB is allocated by PyTorch, and 1.96 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)